In [ ]:
option_quotes

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import newton

# Define the symbol and option types
symbol = 'AAPL'
option_types = ['call', 'put']

# Define the strike prices to generate options for
strike_prices = np.arange(100, 200, 10)

# Define the expiration dates to generate options for
expiration_dates = pd.date_range(start='2020-06-01', end='2022-06-01', freq='D')

# Define the risk-free interest rate and dividend yield
r = 0.02
q = 0.01

# Define the Black-Scholes model for option pricing
def black_scholes(type, s, k, t, r, q, sigma):
    d1 = (np.log(s / k) + (r - q + sigma ** 2 / 2) * t) / (sigma * np.sqrt(t))
    d2 = d1 - sigma * np.sqrt(t)
    if type == 'call':
        price = s * np.exp(-q * t) * norm.cdf(d1) - k * np.exp(-r * t) * norm.cdf(d2)
    else:
        price = k * np.exp(-r * t) * norm.cdf(-d2) - s * np.exp(-q * t) * norm.cdf(-d1)
    return price

# Define the function for implied volatility using the Black-Scholes model
def implied_volatility(type, price, s, k, t, r, q):
    f = lambda sigma: black_scholes(type, s, k, t, r, q, sigma) - price
    return newton(f, 0.5)

# Initialize an empty DataFrame to store the simulated option data
option_quotes = pd.DataFrame(columns=['Symbol', 'Expiration', 'Type', 'Strike', 'Last', 'Change', 'Bid', 'Ask', 'Volume', 'Open Interest', 'Implied Volatility'])

# Loop through each expiration date and strike price to generate option quotes
for expiration in expiration_dates:
    for option_type in option_types:
        for strike_price in strike_prices:
            # Generate random values for each field
            last = np.random.uniform(0, 10)
            change = np.random.uniform(-1, 1)
            bid = np.random.uniform(last - 0.5, last)
            ask = np.random.uniform(last, last + 0.5)
            volume = np.random.randint(0, 1000)
            open_interest = np.random.randint(0, 5000)
            # Compute the implied volatility
            t = (expiration - pd.Timestamp.now()).days / 365
            price = (bid + ask) / 2
            implied_vol = implied_volatility(option_type, price, 100, strike_price, t, r, q)
            # Add the option quote to the DataFrame
            option_quotes = option_quotes.append({
                'Symbol': symbol,
                'Expiration': expiration,
                'Type': option_type,
                'Strike': strike_price,
                'Last': last,
                'Change': change,
                'Bid': bid,
                'Ask': ask,
                'Volume': volume,
                'Open Interest': open_interest,
                'Implied Volatility': implied_vol
            }, ignore_index=True)

# Print the first 10 rows of the option quote chain
print(option_quotes.head(10))


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from py_vollib.black_scholes_merton.implied_volatility import implied_volatility#, bsm

# Define the symbol and option types
symbol = 'AAPL'
option_types = ['call', 'put']

# Define the strike prices to generate options for
strike_prices = np.arange(100, 200, 10)

# Define the expiration dates to generate options for
expiration_dates = pd.date_range(start='2020-06-01', end='2022-06-01', freq='D')

# Initialize an empty DataFrame to store the simulated option data
option_quotes = pd.DataFrame(columns=['Symbol', 'Expiration', 'Type', 'Strike', 'Last', 'Change', 'Bid', 'Ask', 'Volume', 'Open Interest', 'Implied Volatility'])

# Loop through each expiration date and strike price to generate option quotes
for expiration in expiration_dates:
    for option_type in option_types:
        for strike_price in strike_prices:
            # Generate random values for each field except for last
            bid = np.random.uniform(strike_price - 10, strike_price - 1)
            ask = np.random.uniform(strike_price + 1, strike_price + 10)
            volume = np.random.randint(0, 1000)
            open_interest = np.random.randint(0, 5000)
            
            # Generate the last value using the Black-Scholes model
            underlying_price = np.random.uniform(strike_price - 10, strike_price + 10)
            risk_free_rate = 0.02
            time_to_expiry = (expiration - pd.Timestamp.today()).days / 365
            d1 = (np.log(underlying_price/strike_price) + (risk_free_rate + 0.5 * 0.2**2) * time_to_expiry) / (0.2 * np.sqrt(time_to_expiry))
            d2 = d1 - 0.2 * np.sqrt(time_to_expiry)
            if option_type == 'call':
                intrinsic_value = max(0, underlying_price - strike_price)
                last = intrinsic_value + (underlying_price * np.exp(-0.02 * time_to_expiry) * norm.cdf(d1) - strike_price * np.exp(-0.02 * time_to_expiry) * norm.cdf(d2))
            else:
                intrinsic_value = max(0, strike_price - underlying_price)
                last = intrinsic_value + (strike_price * np.exp(-0.02 * time_to_expiry) * norm.cdf(-d2) - underlying_price * np.exp(-0.02 * time_to_expiry) * norm.cdf(-d1))
            
            # Calculate the implied volatility using the Black-Scholes model
           

            
            # Generate the bid and ask prices based on the implied volatility
            try:
                implied_vol = implied_volatility(price=last, S=underlying_price, strike=strike_price, t=time_to_expiry, r=risk_free_rate, flag=option_type)
                bid = np.random.normal(last - 0.1 * last, 0.02 * underlying_price * implied_vol)
                ask = np.random.normal(last + 0.1 * last, 0.02 * underlying_price * implied_vol)
            except:# BelowIntrinsicException:
                implied_vol = np.nan
                bid = np.nan
                ask = np.nan
                
            # Append the option quote to the DataFrame
                
            # Append the generated option quote to the DataFrame
            option_quotes = option_quotes.append({
                'Symbol': symbol,
                'Expiration': expiration,
                'Type': option_type,
                'Strike': strike_price,
                'Last': last,
                'Change': np.nan,
                'Bid': bid,
                'Ask': ask,
                'Volume': volume,
                'Open Interest': open_interest,
                'Implied Volatility': implied_vol
                }, ignore_index=True)

# Print the first 10 rows of the option quotes DataFrame
print(option_quotes.head(10))


In [ ]:
pd.DataFrame(data=option_quotes)

In [ ]:
import numpy as np
import pandas as pd
from py_vollib.black_scholes_merton.implied_volatility import implied_volatility, BelowIntrinsicException

# Define the symbol and option types
symbol = 'AAPL'
option_types = ['call', 'put']

# Define the strike prices to generate options for
strike_prices = np.arange(100, 200, 10)

# Define the expiration dates to generate options for
expiration_dates = pd.date_range(start='2020-06-01', end='2022-06-01', freq='D')

# Initialize an empty DataFrame to store the simulated option data
option_quotes = pd.DataFrame(columns=['Symbol', 'Expiration', 'Type', 'Strike', 'Last', 'Change', 'Bid', 'Ask', 'Volume', 'Open Interest', 'Implied Volatility'])

# Loop through each expiration date and strike price to generate option quotes
for expiration in expiration_dates:
    for option_type in option_types:
        for strike_price in strike_prices:
            # Generate random values for each field
            last = np.random.uniform(0, 10)
            change = np.random.uniform(-1, 1)
            bid = np.random.uniform(last - 0.5, last)
            ask = np.random.uniform(last, last + 0.5)
            volume = np.random.randint(0, 1000)
            open_interest = np.random.randint(0, 5000)
            # Calculate the implied volatility using the Black-Scholes model
            underlying_price = np.random.uniform(120, 130)
            risk_free_rate = 0.02
            time_to_expiry = (expiration - pd.Timestamp.today()).days / 365
            intrinsic_value = max(0, underlying_price - strike_price) if option_type == 'call' else max(0, strike_price - underlying_price)
            if last > intrinsic_value:
                try:
                    implied_vol = implied_volatility(price=last, S=underlying_price, strike=strike_price,
                                                     t=time_to_expiry, r=risk_free_rate, flag=option_type[0], q=0)
                except BelowIntrinsicException:
                    implied_vol = np.nan
                # Add the option quote to the DataFrame
                option_quotes = option_quotes.append({
                    'Symbol': symbol,
                    'Expiration': expiration,
                    'Type': option_type,
                    'Strike': strike_price,
                    'Last': last,
                    'Change': change,
                    'Bid': bid,
                    'Ask': ask,
                    'Volume': volume,
                    'Open Interest': open_interest,
                    'Implied Volatility': implied_vol
                }, ignore_index=True)

# Print the first 10
